In [2]:
from backtesting import Backtest, Strategy
import yfinance as yf
import pandas as pd

/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [3]:
df_msft = yf.download('MSFT', period='1y', interval='4h')
df_msft.columns = df_msft.columns.get_level_values(0)
df_msft.dropna(inplace=True)

[*********************100%***********************]  1 of 1 completed


In [4]:
class MSFT_Strategy(Strategy):
    
    sl_pct = 0.015      # 1.5% stop loss
    tp_pct = 0.025      # 2.5% take profit
    vol_window = 20
    vol_threshold = 1.5
    lookback = 20

    def init(self):
        close = pd.Series(self.data.Close)
        returns = close.pct_change() * 100
        
        self.rolling_vol = self.I(
            lambda x: pd.Series(x).rolling(self.vol_window).std(),
            returns
        )
        
        # Percentil 10 (muy bajo)
        self.rolling_p10 = self.I(
            lambda x: pd.Series(x).rolling(self.lookback).quantile(0.10),
            close
        )

    def next(self):
        price = self.data.Close[-1]
        
        if self.position:
            return
        
        vol = self.rolling_vol[-1]
        avg_vol = self.rolling_vol[-self.vol_window:].mean()
        p10 = self.rolling_p10[-1]
        
        # Entra SOLO si:
        # 1. Volatilidad está ALTA
        # 2. Precio está MUY bajo (percentil 10)
        if vol > avg_vol * self.vol_threshold and price < p10:
            self.buy(
                sl=price * (1 - self.sl_pct),
                tp=price * (1 + self.tp_pct)
            )

In [5]:
bt_msft = Backtest(df_msft, MSFT_Strategy, cash=100000, commission=0.0001)
results_msft = bt_msft.run()
print(results_msft)
print(results_msft['_trades'])

Start                     2025-05-02 13:30...
End                       2026-05-01 17:30...
Duration                    364 days 04:00:00
Exposure Time [%]                     1.60966
Equity Final [$]                 100736.25974
Equity Peak [$]                  103556.51738
Commissions [$]                      81.83624
Return [%]                            0.73626
Buy & Hold Return [%]                -7.99656
Return (Ann.) [%]                      0.7392
Volatility (Ann.) [%]                 3.67345
CAGR [%]                              0.50891
Sharpe Ratio                          0.20123
Sortino Ratio                         0.30133
Calmar Ratio                          0.27143
Alpha [%]                             0.92562
Beta                                  0.02368
Max. Drawdown [%]                     -2.7234
Avg. Drawdown [%]                     -2.7234
Max. Drawdown Duration       91 days 03:00:00
Avg. Drawdown Duration       91 days 03:00:00
# Trades                          

In [6]:
class AAPL_Momentum(Strategy):
    """
    Trend following
    """

    ma_fast = 10
    ma_slow = 20
    sl_pct = 0.015
    tp_pct = 0.025

    def init(self):
        close = pd.Series(self.data.Close)
        self.ma_fast = self.I(lambda x: pd.Series(x).rolling(self.ma_fast).mean(), close)
        self.ma_slow = self.I(lambda x: pd.Series(x).rolling(self.ma_slow).mean(), close)

    def next(self):
        if self.position:
            return
        
        price = self.data.Close[-1]
        
        if self.ma_fast[-1] > self.ma_slow[-1] and self.ma_fast[-2] <= self.ma_slow[-2]:
            self.buy(
                sl=price * (1 - self.sl_pct),
                tp=price * (1 + self.tp_pct)
            )

In [7]:
import sys
sys.path.append('/home/pulpo/Documents/quantitative-trading')
from src.quanttrading.data import prepare_data

df_aapl_4h = prepare_data('AAPL', '1y', '4h')
bt_aapl = Backtest(df_aapl_4h, AAPL_Momentum, cash=100000, commission=0.0001)
results_aapl = bt_aapl.run()
print(results_aapl)
print(results_aapl['_trades'])

[*********************100%***********************]  1 of 1 completed

Start                     2025-05-02 17:30...
End                       2026-05-01 17:30...
Duration                    364 days 00:00:00
Exposure Time [%]                    11.69355
Equity Final [$]                  90700.96216
Equity Peak [$]                  101647.93626
Commissions [$]                     251.75177
Return [%]                           -9.29904
Buy & Hold Return [%]                32.70333
Return (Ann.) [%]                     -9.3343
Volatility (Ann.) [%]                 5.85189
CAGR [%]                             -6.53384
Sharpe Ratio                         -1.59509
Sortino Ratio                        -1.93497
Calmar Ratio                         -0.86673
Alpha [%]                           -11.62923
Beta                                  0.07125
Max. Drawdown [%]                    -10.7695
Avg. Drawdown [%]                    -4.30364
Max. Drawdown Duration      281 days 04:00:00
Avg. Drawdown Duration      103 days 03:00:00
# Trades                          


/tmp/ipykernel_13441/756322505.py:7: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  results_aapl = bt_aapl.run()


In [12]:
class AAPL_by_me(Strategy):
    sl_pct = 0.015
    tp_pct = 0.04
    window = 30

    def init(self):
        close = pd.Series(self.data.Close)
        returns = close.pct_change() * 100

        self.rolling_std = self.I(lambda x: pd.Series(x).rolling(self.window).std(), close)

    def next(self):
        if self.position:
            return
        
        price = self.data.Close[-1]

        if self.rolling_std > 2.3:
            self.open_long(price)

    def open_long(self, price):
        self.buy(
            sl=price * (1 - self.sl_pct),
            tp=price * (1 + self.tp_pct)
        )

In [13]:
bt_aapl_by_me = Backtest(df_aapl_4h, AAPL_by_me, cash=100000, commission=0.0001)
results_aapl_by_me = bt_aapl_by_me.run()
print(results_aapl_by_me)

print(results_aapl_by_me['_trades'])

Start                     2025-05-02 17:30...
End                       2026-05-01 17:30...
Duration                    364 days 00:00:00
Exposure Time [%]                     92.1371
Equity Final [$]                 147536.90419
Equity Peak [$]                   151591.2633
Commissions [$]                    1409.30669
Return [%]                            47.5369
Buy & Hold Return [%]                42.48296
Return (Ann.) [%]                    47.76568
Volatility (Ann.) [%]                29.93155
CAGR [%]                             30.89746
Sharpe Ratio                          1.59583
Sortino Ratio                         4.13624
Calmar Ratio                          3.05379
Alpha [%]                            15.85292
Beta                                   0.7458
Max. Drawdown [%]                   -15.64144
Avg. Drawdown [%]                    -1.94348
Max. Drawdown Duration      149 days 23:00:00
Avg. Drawdown Duration       11 days 04:00:00
# Trades                          

/tmp/ipykernel_13441/1657033573.py:2: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  results_aapl_by_me = bt_aapl_by_me.run()


In [17]:
results_aapl_by_me['_equity_curve']

,Equity,DrawdownPct,DrawdownDuration
Datetime,,,
2025-05-02 17:30:00+00:00,100000.000000,0.000000,NaT
2025-05-05 13:30:00+00:00,100000.000000,0.000000,NaT
2025-05-05 17:30:00+00:00,100000.000000,0.000000,NaT
2025-05-06 13:30:00+00:00,100000.000000,0.000000,NaT
2025-05-06 17:30:00+00:00,100000.000000,0.000000,NaT
...,...,...,...
2026-04-29 17:30:00+00:00,143033.109675,0.056455,NaT
2026-04-30 13:30:00+00:00,144405.865321,0.047400,NaT
2026-04-30 17:30:00+00:00,143652.030635,0.052373,NaT
